In [ ]:
from google.colab import drive
import rasterio
from PIL import Image
from pathlib import Path
import xml.etree.ElementTree as ET
import shutil
import random

In [ ]:
# Connect google drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [ ]:
# Tiles path
tif_dir = Path("/content/gdrive/MyDrive/ParkingSpaceDetection/data/tiles")
png_dir = Path("images")
png_dir.mkdir(parents=True, exist_ok=True)

# Load tiles and convert the tif to png images (necessary for yolo)
for tif_path in tif_dir.glob("*.tif"):
    with rasterio.open(tif_path) as src:
        img = src.read([1, 2, 3])
        img = img.transpose(1, 2, 0)
        out_path = png_dir / (tif_path.stem + ".png")
        Image.fromarray(img).save(out_path)

In [ ]:
# Width and height of image
IMG_W = 1024
IMG_H = 1024

# Assign class
class_map = {"parking_space": 0, "1": 0}

# Labels path
xml_dir = Path("/content/gdrive/MyDrive/ParkingSpaceDetection/data/labels")
yolo_dir = Path("labels")
yolo_dir.mkdir(exist_ok=True)

# Load labels (xml files)
for xml_file in xml_dir.glob("*.xml"):
    tree = ET.parse(xml_file)
    root = tree.getroot()

    yolo_lines = []

    # Extract the bounding box values
    for obj in root.findall("object"):
        cls_name = obj.find("name").text

        if cls_name in class_map:
            cls_id = class_map[cls_name]

            bbox = obj.find("bndbox")
            xmin = float(bbox.find("xmin").text)
            ymin = float(bbox.find("ymin").text)
            xmax = float(bbox.find("xmax").text)
            ymax = float(bbox.find("ymax").text)

            w = xmax - xmin
            h = ymax - ymin
            x_center = xmin + w / 2
            y_center = ymin + h / 2

            yolo_line = f"{cls_id} {x_center/IMG_W:.6f} {y_center/IMG_H:.6f} {w/IMG_W:.6f} {h/IMG_H:.6f}"
            yolo_lines.append(yolo_line)
        else:
            print(f"Warning: Class name '{cls_name}' not found in class_map for file {xml_file.name}.")

    # Convert to text files (necessary for yolo)
    txt_path = yolo_dir / (xml_file.stem + ".txt")
    txt_path.write_text("\n".join(yolo_lines))

In [ ]:
# Paths
img_dir = Path("images")
lbl_dir = Path("labels")

train_img_dir = Path("dataset/images/train")
val_img_dir   = Path("dataset/images/val")
train_lbl_dir = Path("dataset/labels/train")
val_lbl_dir   = Path("dataset/labels/val")

# Create folders
for d in [train_img_dir, val_img_dir, train_lbl_dir, val_lbl_dir]:
    d.mkdir(parents=True, exist_ok=True)

# List images
images = list(img_dir.glob("*.png"))

# Split data in training (80%) and validation (20%)
split_idx = int(0.8 * len(images))
train_imgs = images[:split_idx]
val_imgs   = images[split_idx:]

# Function for moving files to directories
def move_pairs(img_list, target_img_dir, target_lbl_dir):
    for img_path in img_list:
        lbl_path = lbl_dir / (img_path.stem + ".txt")
        shutil.copy(img_path, target_img_dir / img_path.name)
        shutil.copy(lbl_path, target_lbl_dir / lbl_path.name)

# Move files
move_pairs(train_imgs, train_img_dir, train_lbl_dir)
move_pairs(val_imgs,   val_img_dir,   val_lbl_dir)


In [ ]:
pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 32.1 MB/s eta 0:00:00


In [ ]:
from ultralytics import YOLO

# Load pre-trained weights for feature extraction
model = YOLO("yolo11s.pt")

# Load yaml file from drive
yaml = "/content/gdrive/MyDrive/ParkingSpaceDetection/data.yaml"

# Train model
results = model.train(
    data=yaml,
    epochs=100,
    imgsz=1024,
    batch=16,
    device=0,
    single_cls=True,
    rect=True,
    patience=20
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.63 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/gdrive/MyDrive/ParkingSpaceDetection/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015,

In [ ]:
# Validate model
results = model.val(data=yaml)

Ultralytics 8.4.63 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11s summary (fused): 101 layers, 9,413,187 parameters, 0 gradients, 21.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 4479.2±1181.1 MB/s, size: 1327.2 KB)
val: Scanning /content/dataset/labels/val.cache... 25 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 25/25 6.2Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 1.8s/it 3.6s
                   all         25        275      0.852      0.778      0.845      0.481
Speed: 26.5ms preprocess, 28.3ms inference, 0.0ms loss, 2.6ms postprocess per image
Results saved to /content/runs/detect/val


### Visualize Predictions vs. Ground Truth (From Gemini)

This section will iterate through the validation images, run the trained YOLO model to get predictions, load the corresponding ground truth labels, and then display the images with both predicted and ground truth bounding boxes drawn on them for visual comparison.

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np
from ultralytics import YOLO
from pathlib import Path

# Load the best trained model
# Assuming the best model was saved by the previous training run
model = YOLO('/content/runs/detect/train/weights/best.pt')

# Define the paths for images and labels
val_img_dir = Path('dataset/images/val')
val_lbl_dir = Path('dataset/labels/val')

# Get the list of validation image paths (from kernel state)
# Or re-create it if the kernel state is not persistent or for robustness
# Assuming 'val_imgs' from previous cell 3JuKFbUgj4Km is available or re-deriving
img_dir = Path('images')
images_all = list(img_dir.glob('*.png'))
split_idx = int(0.9 * len(images_all))
val_imgs_paths = images_all[split_idx:]

# Helper function to draw bounding boxes
def draw_boxes(image, boxes, color, label=None):
    for box in boxes:
        x1, y1, x2, y2 = map(int, box[:4])
        cv2.rectangle(image, (x1, y1), (x2, y2), color, 2) # Draw rectangle
        if label is not None:
            # Add label text (e.g., 'GT' or 'Pred')
            cv2.putText(image, label, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)

# Iterate through validation images
plt.figure(figsize=(20, 10 * len(val_imgs_paths)))

for i, img_path in enumerate(val_imgs_paths):
    # Construct full path to validation image
    actual_img_path = val_img_dir / img_path.name

    # Read the image
    img = cv2.imread(str(actual_img_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB) # Convert BGR to RGB for matplotlib

    # Get predictions from the model
    results = model.predict(img, save=False, verbose=False, conf=0.25) # conf=0.25 is default
    pred_boxes = []
    if results and len(results) > 0:
        for *xyxy, conf, cls in results[0].boxes.data:
            pred_boxes.append(xyxy)

    # Load ground truth labels
    lbl_file = val_lbl_dir / (img_path.stem + '.txt')
    gt_boxes = []
    if lbl_file.exists():
        with open(lbl_file, 'r') as f:
            for line in f.readlines():
                # YOLO format: class_id center_x center_y width height (normalized)
                parts = list(map(float, line.strip().split()))
                if len(parts) == 5:
                    class_id, cx, cy, w, h = parts
                    # Convert normalized (cx, cy, w, h) to (x1, y1, x2, y2)
                    img_h, img_w, _ = img.shape
                    x1 = (cx - w/2) * img_w
                    y1 = (cy - h/2) * img_h
                    x2 = (cx + w/2) * img_w
                    y2 = (cy + h/2) * img_h
                    gt_boxes.append([x1, y1, x2, y2])

    # Draw ground truth boxes (Green)
    draw_boxes(img, gt_boxes, (0, 255, 0), label='GT')
    # Draw predicted boxes (Red)
    draw_boxes(img, pred_boxes, (255, 0, 0), label='Pred')

    # Display the image
    plt.subplot(len(val_imgs_paths), 1, i + 1)
    plt.imshow(img)
    plt.title(f'Image: {img_path.name} (GT: Green, Pred: Red)')
    plt.axis('off')

plt.tight_layout()
plt.show()

Output hidden; open in https://colab.research.google.com to view.